# 01 · Exploratory Data Analysis — Kerala 2018 + Sen1Floods11

**Goal:** sanity-check the Phase 1 data artefacts before committing to the Phase 2 DIP pipeline.

Runs cleanly on **Google Colab** (GPU runtime not required for this notebook).

What this notebook covers:
1. Load pre- and post-event Sentinel-2 composites for Kerala.
2. Visualise RGB + false-colour (NIR-R-G) composites.
3. Band-wise reflectance histograms & summary stats.
4. Scene-level cloud-cover inspection via the SCL band (if present).
5. Ground-truth flood mask overlay.
6. A handful of Sen1Floods11 chips with labels.

**Pre-requisites** (run once in the repo):
- `python -m src.data.gee_download --config configs/kerala_2018.yaml --window both --out-dir data/raw/kerala_2018 --project <your-gcp-project>`
- `python scripts/download_sen1floods11.py --dest /content/drive/MyDrive/dda/sen1floods11 --subset hand`

## 0 · Setup

In [ ]:
# If running on Colab, uncomment:
# !git clone https://github.com/bhagya2819/Disaster_Damage_Assessment.git
# %cd Disaster_Damage_Assessment
# !pip install -q -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')

import sys, os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

# Make sure the repo root is on sys.path so `import src.*` works when the
# notebook is opened from notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / 'PRD.md').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.aoi import load_aoi
from src.data.sen1floods11_loader import Sen1Floods11Dataset

aoi = load_aoi('configs/kerala_2018.yaml')
print(aoi.name, '→', aoi.bbox, aoi.crs)

## 1 · Load pre/post Sentinel-2 composites

In [ ]:
PRE  = Path('data/raw/kerala_2018/kerala_2018_pre.tif')
POST = Path('data/raw/kerala_2018/kerala_2018_post.tif')

for p in (PRE, POST):
    assert p.exists(), f'Missing: {p} — run src.data.gee_download first.'

def read_bands(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
        return arr, src.profile

pre_arr,  pre_profile  = read_bands(PRE)
post_arr, post_profile = read_bands(POST)
print('pre  shape:', pre_arr.shape,  'CRS:', pre_profile['crs'])
print('post shape:', post_arr.shape, 'CRS:', post_profile['crs'])
print('bands (config order):', aoi.bands)

## 2 · True-colour + false-colour composites

- **True colour:** B4 (red), B3 (green), B2 (blue).
- **False colour (NIR-R-G):** B8, B4, B3 — vegetation glows red, water absorbs NIR → appears very dark.

In [ ]:
def stretch(x, lo=2, hi=98):
    """Per-band percentile stretch for display."""
    out = np.zeros_like(x, dtype=np.float32)
    for i in range(x.shape[0]):
        a, b = np.percentile(x[i], [lo, hi])
        out[i] = np.clip((x[i] - a) / (b - a + 1e-9), 0, 1)
    return out

def rgb(arr, band_ids):
    # band_ids are config indices (0-based) corresponding to aoi.bands order
    return np.transpose(stretch(arr[band_ids]), (1, 2, 0))

# aoi.bands = ['B2','B3','B4','B8','B11','B12'] → indices
TRUE    = [2, 1, 0]   # B4, B3, B2
NIR_FC  = [3, 2, 1]   # B8, B4, B3

fig, axes = plt.subplots(2, 2, figsize=(11, 11))
axes[0, 0].imshow(rgb(pre_arr,  TRUE));    axes[0, 0].set_title('Pre-event · true colour');   axes[0, 0].axis('off')
axes[0, 1].imshow(rgb(post_arr, TRUE));    axes[0, 1].set_title('Post-event · true colour');  axes[0, 1].axis('off')
axes[1, 0].imshow(rgb(pre_arr,  NIR_FC));  axes[1, 0].set_title('Pre-event · NIR-R-G');       axes[1, 0].axis('off')
axes[1, 1].imshow(rgb(post_arr, NIR_FC));  axes[1, 1].set_title('Post-event · NIR-R-G');      axes[1, 1].axis('off')
plt.tight_layout(); plt.savefig('reports/figures/eda_composites.png', dpi=150, bbox_inches='tight'); plt.show()

## 3 · Band reflectance histograms (pre vs post)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (i, band) in zip(axes.ravel(), enumerate(aoi.bands)):
    ax.hist(pre_arr[i].ravel(),  bins=80, alpha=0.55, label='pre',  density=True)
    ax.hist(post_arr[i].ravel(), bins=80, alpha=0.55, label='post', density=True)
    ax.set_title(band); ax.set_yscale('log'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('reports/figures/eda_hist.png', dpi=150, bbox_inches='tight'); plt.show()

## 4 · Summary statistics

Expect post-event reflectance in NIR (B8) to drop over flooded pixels (water absorbs NIR) and SWIR (B11/B12) likewise to drop. Any band whose mean barely changes is a candidate to **exclude** from difference indices.

In [ ]:
import pandas as pd
rows = []
for i, band in enumerate(aoi.bands):
    rows.append({
        'band':  band,
        'pre_mean':  float(pre_arr[i].mean()),
        'post_mean': float(post_arr[i].mean()),
        'delta':     float(post_arr[i].mean() - pre_arr[i].mean()),
        'pre_std':   float(pre_arr[i].std()),
        'post_std':  float(post_arr[i].std()),
    })
pd.DataFrame(rows).round(4)

## 5 · Ground-truth overlay (UNOSAT)

Run `src.data.ground_truth.rasterize_flood_polygons` once to produce `data/gt/kerala_gt.tif`. Cell is skipped gracefully if the mask is not yet on disk.

In [ ]:
GT = Path('data/gt/kerala_gt.tif')
if GT.exists():
    with rasterio.open(GT) as src:
        gt = src.read(1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(rgb(post_arr, TRUE)); ax[0].set_title('Post-event'); ax[0].axis('off')
    ax[1].imshow(rgb(post_arr, TRUE)); ax[1].imshow(gt, cmap='Blues', alpha=0.45)
    ax[1].set_title(f'GT overlay · flood fraction = {gt.mean():.3f}'); ax[1].axis('off')
    plt.tight_layout(); plt.savefig('reports/figures/eda_gt_overlay.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print('No GT mask yet — run rasterize_flood_polygons(...) once you have the UNOSAT shapefile.')

## 6 · Peek at Sen1Floods11 training chips

In [ ]:
SEN1F_ROOT = Path(os.environ.get('SEN1FLOODS11_DIR',
                                 '/content/drive/MyDrive/dda/sen1floods11'))
if SEN1F_ROOT.exists():
    ds = Sen1Floods11Dataset(SEN1F_ROOT, split='train', modality='s2')
    print('train split size:', len(ds))
    ids = np.random.default_rng(0).choice(len(ds), size=4, replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for col, i in enumerate(ids):
        s = ds[int(i)]
        img = s['image'].numpy()[[2, 1, 0]]  # B4, B3, B2
        axes[0, col].imshow(np.transpose(stretch(img), (1, 2, 0)))
        axes[0, col].set_title(s['chip_id'], fontsize=8); axes[0, col].axis('off')
        axes[1, col].imshow(s['label'].numpy(), cmap='Blues', vmin=-1, vmax=1)
        axes[1, col].axis('off')
    plt.tight_layout(); plt.savefig('reports/figures/eda_sen1floods11.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print(f'Sen1Floods11 not found at {SEN1F_ROOT}. Run scripts/download_sen1floods11.py first.')

## 7 · Phase 1 exit-criteria checklist

- [ ] `data/raw/kerala_2018/kerala_2018_{pre,post}.tif` exist, valid CRS, non-zero reflectance.
- [ ] `data/gt/kerala_gt.tif` exists and aligns with the post raster.
- [ ] Sen1Floods11 HandLabeled tree readable; train split returns > 0 chips.
- [ ] `pytest tests/test_data.py` green.

When every box is ticked, proceed to **Phase 2 — Classical DIP**.